
# Landing to Bronze — V-Credit

## Objetivo

Ingerir os oito arquivos CSV de origem e armazená-los em tabelas Delta na camada Bronze do catálogo `vcredit`.

## Responsabilidades

- Criar e verificar a estrutura inicial do projeto.
- Conferir a presença dos arquivos de origem.
- Ler os CSVs com a codificação definida para cada arquivo.
- Preservar os valores de origem, mantendo as colunas como texto.
- Adicionar `ingestion_datetime` e `arquivo_origem` para rastreabilidade.
- Validar as quantidades de registros após a gravação.


In [0]:
# 1. Criar o catálogo do projeto
spark.sql("""
    CREATE CATALOG IF NOT EXISTS vcredit
""")

# 2. Criar os schemas das três camadas
for schema in ["bronze", "silver", "gold"]:
    spark.sql(f"""
        CREATE SCHEMA IF NOT EXISTS vcredit.{schema}
    """)

# 3. Criar o volume de entrada na camada Bronze
spark.sql("""
    CREATE VOLUME IF NOT EXISTS vcredit.bronze.inputs
""")

# 4. Definir o caminho dos arquivos de origem
input_path = "/Volumes/vcredit/bronze/inputs/"

In [0]:
# 5. Listar os arquivos disponíveis
arquivos_disponiveis = dbutils.fs.ls(input_path)
display(arquivos_disponiveis)

# 6. Conferir se os oito arquivos esperados estão presentes
arquivos_esperados = [
    "base_atendentes.csv",
    "base_motivos.csv",
    "canais.csv",
    "chamados.csv",
    "Chamados_Hora.CSV",
    "clientes.csv",
    "custos.csv",
    "pesquisa_satisfacao.csv"
]

nomes_disponiveis = {
    arquivo.name for arquivo in arquivos_disponiveis
}

arquivos_faltantes = [
    nome for nome in arquivos_esperados
    if nome not in nomes_disponiveis
]

if arquivos_faltantes:
    raise FileNotFoundError(
        "Arquivos ausentes no volume de entrada: "
        + ", ".join(arquivos_faltantes)
    )

print("Todos os oito arquivos de origem foram encontrados.")

path,name,size,modificationTime
dbfs:/Volumes/vcredit/bronze/inputs/Chamados_Hora.CSV,Chamados_Hora.CSV,11547467,1793035943000
dbfs:/Volumes/vcredit/bronze/inputs/base_atendentes.csv,base_atendentes.csv,1525,1793035939000
dbfs:/Volumes/vcredit/bronze/inputs/base_motivos.csv,base_motivos.csv,800,1793035939000
dbfs:/Volumes/vcredit/bronze/inputs/canais.csv,canais.csv,219,1793035939000
dbfs:/Volumes/vcredit/bronze/inputs/chamados.csv,chamados.csv,14825050,1793035944000
dbfs:/Volumes/vcredit/bronze/inputs/clientes.csv,clientes.csv,4568046,1793035941000
dbfs:/Volumes/vcredit/bronze/inputs/custos.csv,custos.csv,3343365,1793035941000
dbfs:/Volumes/vcredit/bronze/inputs/pesquisa_satisfacao.csv,pesquisa_satisfacao.csv,1705822,1793035940000


Todos os oito arquivos de origem foram encontrados.



## Função de ingestão

A função `carregar_bronze` centraliza a leitura, as validações e a gravação dos arquivos na camada Bronze.

As colunas de origem são mantidas como texto. A função acrescenta metadados de rastreabilidade, grava os dados em formato Delta e compara as quantidades de registros lidos e gravados.

In [0]:
from pyspark.sql import functions as F


def carregar_bronze(nome_arquivo, nome_tabela, encoding):

    # 1. Montar o caminho completo do arquivo
    caminho = input_path + nome_arquivo

    print(f"Iniciando leitura: {nome_arquivo}")
    print(f"Codificação utilizada: {encoding}")

    # 2. Ler o arquivo CSV
    df = (
        spark.read
        .option("header", True)
        .option("inferSchema", False)
        .option("sep", ",")
        .option("encoding", encoding)
        .option("multiLine", True)
        .option("escape", '"')
        .csv(caminho)
    )

    # 3. Padronizar os nomes das colunas
    df = df.toDF(
        *[coluna.strip().lower() for coluna in df.columns]
    )

    # 4. Validar se o arquivo possui colunas
    if len(df.columns) == 0:
        raise ValueError(
            f"O arquivo {nome_arquivo} não possui colunas."
        )

    # 5. Validar nomes duplicados após a padronização
    if len(df.columns) != len(set(df.columns)):
        raise ValueError(
            f"O arquivo {nome_arquivo} possui "
            "nomes de colunas duplicados."
        )

    # 6. Conferir a estrutura e a quantidade de registros
    df.printSchema()

    quantidade_origem = df.count()

    print(
        f"Registros lidos de {nome_arquivo}: "
        f"{quantidade_origem}"
    )

    # 7. Adicionar metadados de rastreabilidade
    df = (
        df
        .withColumn(
            "ingestion_datetime",
            F.current_timestamp()
        )
        .withColumn(
            "arquivo_origem",
            F.lit(nome_arquivo)
        )
    )

    # 8. Gravar na tabela Bronze
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(nome_tabela)
    )

    # 9. Ler a tabela gravada para validar o resultado
    df_bronze = spark.table(nome_tabela)

    quantidade_bronze = df_bronze.count()

    print(f"Tabela carregada: {nome_tabela}")
    print(f"Registros de origem: {quantidade_origem}")
    print(f"Registros na Bronze: {quantidade_bronze}")

    # 10. Verificar se as quantidades correspondem
    if quantidade_origem != quantidade_bronze:
        raise ValueError(
            f"A quantidade de registros não corresponde "
            f"na tabela {nome_tabela}."
        )

    print("Carga concluída com sucesso.")

    return df_bronze



## Mapeamento dos arquivos de origem

Nesta etapa, cada CSV é associado à sua tabela Delta na camada Bronze e à codificação definida para a leitura.

O processamento é executado por um laço, reutilizando a função de ingestão para as oito bases.

In [0]:

# Arquivo de origem -> tabela Bronze -> codificação
arquivos_bronze = [
    (
        "chamados.csv",
        "vcredit.bronze.tb_chamados",
        "UTF-8"
    ),
    (
        "Chamados_Hora.CSV",
        "vcredit.bronze.tb_chamados_hora",
        "ISO-8859-1"
    ),
    (
        "clientes.csv",
        "vcredit.bronze.tb_clientes",
        "ISO-8859-1"
    ),
    (
        "custos.csv",
        "vcredit.bronze.tb_custos",
        "UTF-8"
    ),
    (
        "pesquisa_satisfacao.csv",
        "vcredit.bronze.tb_pesquisa_satisfacao",
        "UTF-8"
    ),
    (
        "canais.csv",
        "vcredit.bronze.tb_canais",
        "UTF-8"
    ),
    (
        "base_motivos.csv",
        "vcredit.bronze.tb_motivos",
        "UTF-8"
    ),
    (
        "base_atendentes.csv",
        "vcredit.bronze.tb_atendentes",
        "UTF-8"
    )
]

# Executar a carga de cada arquivo
for nome_arquivo, nome_tabela, encoding in arquivos_bronze:
    carregar_bronze(
        nome_arquivo,
        nome_tabela,
        encoding
    )


Iniciando leitura: chamados.csv
Codificação utilizada: UTF-8
root
 |-- id_chamado: string (nullable = true)
 |-- id_cliente: string (nullable = true)
 |-- motivo: string (nullable = true)
 |-- canal: string (nullable = true)
 |-- resolvido: string (nullable = true)
 |-- hora_abertura_chamado: string (nullable = true)
 |-- hora_inicio_atendimento: string (nullable = true)
 |-- hora_finalizacao_atendimento: string (nullable = true)
 |-- tempo_espera: string (nullable = true)
 |-- tempo_atendimento: string (nullable = true)
 |-- id_atendente: string (nullable = true)

Registros lidos de chamados.csv: 134474
Tabela carregada: vcredit.bronze.tb_chamados
Registros de origem: 134474
Registros na Bronze: 134474
Carga concluída com sucesso.
Iniciando leitura: Chamados_Hora.CSV
Codificação utilizada: ISO-8859-1
root
 |-- id_chamado: string (nullable = true)
 |-- id_cliente: string (nullable = true)
 |-- hora_abertura_chamado: string (nullable = true)
 |-- hora_inicio_atendimento: string (nullabl


## Validação das tabelas Bronze

Após a carga, este bloco consulta as tabelas gravadas, exibe a quantidade de registros e apresenta uma amostra dos dados para inspeção.


In [0]:
# Verificar as tabelas carregadas na camada Bronze
for _, nome_tabela, _ in arquivos_bronze:
    df_bronze = spark.table(nome_tabela)

    print(f"\nTabela: {nome_tabela}")
    print(
        f"Quantidade de registros: {df_bronze.count()}"
    )

    display(df_bronze.limit(5))


Tabela: vcredit.bronze.tb_chamados
Quantidade de registros: 134474


id_chamado,id_cliente,motivo,canal,resolvido,hora_abertura_chamado,hora_inicio_atendimento,hora_finalizacao_atendimento,tempo_espera,tempo_atendimento,id_atendente,ingestion_datetime,arquivo_origem
76696,74719850215,Renegociação de dívida,atendimento inicial,Sim,null,null,null,49,441,4,2026-10-28T18:27:39.789Z,chamados.csv
44377,92983951302,CONSULTA DE CONTRATO,Atendimento_Inicial,Sim,null,null,null,374,355,8,2026-10-28T18:27:39.789Z,chamados.csv
34306,77740207578,Consulta de limite,ATENDIMENTO INICIAL,Sim,null,null,null,110,985,6,2026-10-28T18:27:39.789Z,chamados.csv
52060,36276971799,Renegociação de dívida,Ura,Não,null,igual a hora de abertura,null,NULL,216,9,2026-10-28T18:27:39.789Z,chamados.csv
25271,53061261807,Renegociação de dívida,Atendimento_Inicial,N,null,null,null,96,384,2,2026-10-28T18:27:39.789Z,chamados.csv



Tabela: vcredit.bronze.tb_chamados_hora
Quantidade de registros: 131838


id_chamado,id_cliente,hora_abertura_chamado,hora_inicio_atendimento,hora_finalizacao_atendimento,ingestion_datetime,arquivo_origem
121151,25203409261,29/06/2026 às 14:12:48,29/06/2026 às 14:21:24,29/06/2026 às 15:06:16,2026-10-28T18:27:49.086Z,Chamados_Hora.CSV
48388,99076736475,16/09/2025 às 09:03:42,16/09/2025 às 09:07:41,16/09/2025 às 10:11:50,2026-10-28T18:27:49.086Z,Chamados_Hora.CSV
12605,13153799751,03/03/2025 às 14:02:31,03/03/2025 às 14:02:31,03/03/2025 às 14:04:25,2026-10-28T18:27:49.086Z,Chamados_Hora.CSV
120663,13276442322,08/06/2026 às 15:38:20,08/06/2026 às 15:38:20,08/06/2026 às 15:40:18,2026-10-28T18:27:49.086Z,Chamados_Hora.CSV
45377,74032831005,23/09/2025 às 14:41:45,23/09/2025 às 14:41:45,23/09/2025 às 14:43:35,2026-10-28T18:27:49.086Z,Chamados_Hora.CSV



Tabela: vcredit.bronze.tb_clientes
Quantidade de registros: 47705


...

id_canal,nome_canal,tipo_canal,status_canal,ingestion_datetime,arquivo_origem
1,URA,Autosservico,ativo,2026-10-28T18:28:09.201Z,canais.csv
2,CHATBOT,Autosservico,ativo,2026-10-28T18:28:09.201Z,canais.csv
3,ATENDIMENTO INICIAL,Humano,ativo,2026-10-28T18:28:09.201Z,canais.csv
4,ATENDIMENTO ESPECIALIZADO,Humano,ativo,2026-10-28T18:28:09.201Z,canais.csv



Tabela: vcredit.bronze.tb_motivos
Quantidade de registros: 14


id_motivo,nome_motivo,categoria,criticidade,ingestion_datetime,arquivo_origem
11,Bloqueio de cartão,Cartao,Alta,2026-10-28T18:28:13.920Z,base_motivos.csv
7,Compra não autorizada,Cartao,alta,2026-10-28T18:28:13.920Z,base_motivos.csv
13,Dúvidas gerais sobre programa de pontos,Beneficios,baixa,2026-10-28T18:28:13.920Z,base_motivos.csv
14,Transferência de agência,null,Media,2026-10-28T18:28:13.920Z,base_motivos.csv
9,Problema com aplicativo,Atendimento,Media,2026-10-28T18:28:13.920Z,base_motivos.csv



Tabela: vcredit.bronze.tb_atendentes
Quantidade de registros: 25


id_atendente,nome_atendente,nivel_atendimento,data_admissao,data_desligamento,status_atendente,ingestion_datetime,arquivo_origem
1,Marcela França,1,18/04/2019,16/11/2025,inativo,2026-10-28T18:28:18.194Z,base_atendentes.csv
2,Zilda Esteves,1,12/09/2023,24/01/2026,inativo,2026-10-28T18:28:18.194Z,base_atendentes.csv
3,William Marques,1,28/05/2019,15/04/2025,inativo,2026-10-28T18:28:18.194Z,base_atendentes.csv
4,Solange Silveira,Nível 1,07/10/2019,null,ativo,2026-10-28T18:28:18.194Z,base_atendentes.csv
5,Joaquim Barros,Nível 1,06/12/2020,23/05/2026,inativo,2026-10-28T18:28:18.194Z,base_atendentes.csv



## Verificação complementar da codificação

Esta etapa procura padrões de caracteres que podem indicar problemas de leitura ou codificação nas tabelas Bronze.

Os resultados são indícios para investigação e devem ser comparados com os arquivos de origem. Esta verificação não modifica nem corrige os dados.

In [0]:

# Verificar possíveis problemas de codificação nas tabelas Bronze

caracteres_suspeitos = ["�", "Ã§", "Ã£", "Ã¡", "Ã©", "Ã³", "Ãº", "Ã­", "Â"]

for _, nome_tabela, _ in arquivos_bronze:
    df = spark.table(nome_tabela)

    # Selecionar as colunas de texto
    colunas_texto = [
        campo.name
        for campo in df.schema.fields
        if campo.dataType.simpleString() == "string"
    ]

    # Procurar caracteres suspeitos em qualquer coluna de texto
    condicao = None

    for coluna in colunas_texto:
        for caractere in caracteres_suspeitos:
            teste = F.col(coluna).contains(caractere)
            condicao = teste if condicao is None else condicao | teste

    if condicao is not None:
        suspeitos = df.filter(condicao)
        quantidade = suspeitos.count()

        print(f"\nTabela: {nome_tabela}")
        print(f"Linhas com possíveis problemas: {quantidade}")

        if quantidade > 0:
            display(suspeitos.limit(5))
    else:
        print(f"\nTabela: {nome_tabela} - sem colunas de texto")



Tabela: vcredit.bronze.tb_chamados
Linhas com possíveis problemas: 0

Tabela: vcredit.bronze.tb_chamados_hora
Linhas com possíveis problemas: 0

Tabela: vcredit.bronze.tb_clientes
Linhas com possíveis problemas: 0

Tabela: vcredit.bronze.tb_custos
Linhas com possíveis problemas: 0

Tabela: vcredit.bronze.tb_pesquisa_satisfacao
Linhas com possíveis problemas: 0

Tabela: vcredit.bronze.tb_canais
Linhas com possíveis problemas: 0

Tabela: vcredit.bronze.tb_motivos
Linhas com possíveis problemas: 0

Tabela: vcredit.bronze.tb_atendentes
Linhas com possíveis problemas: 0
